# OpenPlaque End-to-End Segmentation

Fresh Colab workflow:

1. Mount Google Drive
2. Clone/update OpenPlaque from GitHub
3. Install Colab requirements
4. Configure nnU-Net paths
5. Load the full DICOM ZIP
6. Load curved LAD/RCA/LCX reformats
7. Run plaque segmentation
8. Compute total plaque volume
9. Save masks to Google Drive

Expected Google Drive files:

```text
/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip
/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip
```


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Clone or update OpenPlaque
!git clone https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque || true
!git -C /content/OpenPlaque pull


In [ ]:
# Install requirements
!wget -q https://raw.githubusercontent.com/pazzani/OpenPlaque/main/requirements-colab.txt -O /content/requirements-colab.txt
!pip install -q -r /content/requirements-colab.txt


In [ ]:
# Configure Python path and nnU-Net environment
import os
import sys
from pathlib import Path

repo = Path("/content/OpenPlaque")
src = repo / "src"

if str(src) not in sys.path:
    sys.path.insert(0, str(src))

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/nnUNet_results"

for d in [os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]]:
    os.makedirs(d, exist_ok=True)

print("OpenPlaque path:", src)
print("nnUNet_results:", os.environ["nnUNet_results"])


In [ ]:
# Extract nnU-Net model weights if needed
import zipfile
from pathlib import Path

model_zip = Path("/content/drive/MyDrive/OpenPlaque/models/Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip")
model_target = Path("/content/nnUNet_results/Dataset001_CCTA_DHM")

if model_target.exists():
    print("Model already extracted:", model_target)
else:
    if not model_zip.exists():
        raise FileNotFoundError(f"Missing model ZIP: {model_zip}")
    print("Extracting model ZIP...")
    with zipfile.ZipFile(model_zip) as z:
        z.extractall("/content/nnUNet_results")
    print("Extracted model to /content/nnUNet_results")

!find /content/nnUNet_results -maxdepth 3


In [ ]:
# Load full DICOM study
from openplaque.study import OpenPlaqueStudy

study_zip = "/content/drive/MyDrive/OpenPlaque/Full_DICOM.zip"
study = OpenPlaqueStudy(study_zip)

study.summary()


## Load curved coronary reformats

For the UCLA study inventory:

- Series 1035 = RCA curved reformat
- Series 1039 = CX/LCX curved reformat
- Series 1043 = LAD curved reformat


In [ ]:
# Load curved reformats
image_rca, volume_rca, _ = study.load_series(1035)
image_lcx, volume_lcx, _ = study.load_series(1039)
image_lad, volume_lad, _ = study.load_series(1043)

print("RCA:", volume_rca.shape, image_rca.GetSpacing())
print("LCX:", volume_lcx.shape, image_lcx.GetSpacing())
print("LAD:", volume_lad.shape, image_lad.GetSpacing())


In [ ]:
# Optional quick visual check
import matplotlib.pyplot as plt

def show_mid(volume, title):
    z = volume.shape[0] // 2
    plt.figure(figsize=(6,6))
    plt.imshow(volume[z], cmap="gray", vmin=-200, vmax=800)
    plt.title(title)
    plt.axis("off")
    plt.show()

show_mid(volume_lad, "LAD curved reformat")
show_mid(volume_rca, "RCA curved reformat")
show_mid(volume_lcx, "LCX curved reformat")


In [ ]:
# Import segmentation module
from openplaque.segmentation import segment_vessel


## Run plaque segmentation

This runs the pretrained `Dataset001_CCTA_DHM` nnU-Net model on each curved artery reformat.

This may take several minutes.


In [ ]:
lad_report = segment_vessel(image_lad, volume_lad, "LAD")
lad_report.summary()
lad_report.show_overlay(label=2)


In [ ]:
rca_report = segment_vessel(image_rca, volume_rca, "RCA")
rca_report.summary()
rca_report.show_overlay(label=2)


In [ ]:
lcx_report = segment_vessel(image_lcx, volume_lcx, "LCX")
lcx_report.summary()
lcx_report.show_overlay(label=2)


## Total Plaque Volume

In [ ]:
reports = [lad_report, rca_report, lcx_report]

total_tpv = sum(r.tpv_mm3 for r in reports)

print("Plaque volume by vessel")
print("-" * 40)

for r in reports:
    print(f"{r.name}: {r.tpv_mm3:.2f} mm³ ({r.plaque_voxels} voxels)")

print("-" * 40)
print(f"TOTAL PLAQUE VOLUME: {total_tpv:.2f} mm³")


In [ ]:
# Save masks to Google Drive
import os

save_dir = "/content/drive/MyDrive/OpenPlaque/Segmentations"
os.makedirs(save_dir, exist_ok=True)

lad_report.save_mask(f"{save_dir}/LAD_plaque_segmentation.nii.gz")
rca_report.save_mask(f"{save_dir}/RCA_plaque_segmentation.nii.gz")
lcx_report.save_mask(f"{save_dir}/LCX_plaque_segmentation.nii.gz")

print("Saved segmentations to:", save_dir)


In [ ]:
# Save a small text summary
summary_path = "/content/drive/MyDrive/OpenPlaque/Segmentations/tpv_summary.txt"

with open(summary_path, "w") as f:
    f.write("OpenPlaque experimental plaque segmentation summary\n")
    f.write("Research use only. Not for clinical decision-making.\n\n")
    for r in reports:
        f.write(f"{r.name}: {r.tpv_mm3:.2f} mm3 ({r.plaque_voxels} voxels)\n")
    f.write(f"TOTAL PLAQUE VOLUME: {total_tpv:.2f} mm3\n")

print("Saved:", summary_path)
